# V109 - No-chat action under exact-text folds

Pre-registered re-verification: v108 showed verdicts move with fold definition, so the adopted no-chat action must survive the harsher exact-text split too. Bagged word base plus bagged assessment override on no-chat rows only, three locks as in v100, fresh seed. A pass keeps the v100 submission standing. A fail withdraws it back to the champion.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold

SEED = 3434
N_SPLITS = 5
SLICE_FLOOR = 0.01
BAGS = 10
rng = np.random.RandomState(SEED)
WORDS = {"nol": 0, "satu": 1, "dua": 2, "tiga": 3, "empat": 4, "lima": 5, "tidak": 0, "ya": 5}

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v109")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve())

## Bagged views, no-chat override, three locks

Identical action to v100, harsher folds. The submission file is touched by nothing here; this run only rules on it.

In [ ]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assess = pd.read_csv(DATA / "user_assessments.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)
all_users = list(train["user_id"]) + test_ids

def num(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return float(WORDS.get(str(x).strip().lower(), np.nan))

blobs = {r["user_id"]: json.loads(r["assessment_result"]) for _, r in assess.iterrows()}
questions = sorted({q for b in blobs.values() for q in b})
A = np.array([[num(blobs[u].get(q)) for q in questions] for u in all_users])
A = np.where(np.isnan(A), np.nanmean(A, axis=0), A)
A = (A - A.mean(0)) / (A.std(0) + 1e-9)
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in all_users]
W = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs).toarray()
W_tr, A_tr = W[:n_train], A[:n_train]
is_nochat = np.array([u not in by_user for u in all_users])
nochat_tr = is_nochat[:n_train]
firsts = [str(by_user.get(u, [""])[0]) if by_user.get(u, []) else "NO_CHAT" for u in list(train["user_id"])]
fold_groups = np.array(firsts)
print(f"exact-text groups: {len(set(firsts))}; no-chat train users: {nochat_tr.sum()}")
strata = Y.argmax(1)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

def bagged_oof(Xtr, seed):
    r = np.random.RandomState(seed)
    acc = np.zeros_like(Y)
    cnt = np.zeros(n_train)
    for b in range(BAGS):
        idx = r.choice(n_train, n_train, replace=True)
        oob = np.setdiff1d(np.arange(n_train), np.unique(idx))
        for j in range(len(MODS)):
            m = Ridge(alpha=1.0)
            m.fit(Xtr[idx], Y[idx, j])
            acc[oob, j] += m.predict(Xtr[oob])
        cnt[oob] += 1
    return acc / np.maximum(cnt[:, None], 1)

oof_bw = bagged_oof(W_tr, SEED)
oof_ba = bagged_oof(A_tr, SEED + 1)
base = ndcg5(Y, oof_bw)
act = oof_bw.copy()
act[nochat_tr] = oof_ba[nochat_tr]
d = ndcg5(Y, act) - base
ds = d[nochat_tr]
lo_s, hi_s = np.quantile([rng.choice(ds, len(ds), replace=True).mean() for _ in range(5000)], [0.025, 0.975])
fids = np.zeros(n_train, int)
skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
for fi, (tr, va) in enumerate(skf.split(W_tr, strata, fold_groups)):
    fids[va] = fi + 1
wf = min(ndcg5(Y[fids == k], act[fids == k]).mean() - base[fids == k].mean() for k in range(1, N_SPLITS + 1))
print(f"base={base.mean():.5f} action={ndcg5(Y, act).mean():.5f} pooled={d.mean():+.5f} slice CI95=[{lo_s:+.5f},{hi_s:+.5f}] worst-fold={wf:+.5f}")
print("SUBMISSION STANDS" if ((lo_s > SLICE_FLOOR) and (wf >= 0)) else "SUBMISSION WITHDRAWN: action fails under exact-text folds")